<a href="https://colab.research.google.com/github/keertiam8/gnn-congestion/blob/main/gnn_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# cell A
import shutil, os
shutil.rmtree("/content/closed-loop-placement", ignore_errors=True)
os.chdir("/content")
!git clone https://github.com/keertiam8/gnn-congestion closed-loop-placement
os.chdir("/content/closed-loop-placement")
!git log --oneline -3

Cloning into 'closed-loop-placement'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 114 (delta 60), reused 75 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 864.91 KiB | 17.65 MiB/s, done.
Resolving deltas: 100% (60/60), done.
e828b4e (HEAD -> main, origin/main, origin/HEAD) Add explicit variance-matching term, lower default peak_weight
99ab360 Switch LayerNorm to GraphNorm, add congestion-weighted loss
dc15b64 Add global skip connection to fix GNN oversmoothing, expose --lr flag


In [8]:
# cell B
!pip install --quiet gdown scipy scikit-image


In [4]:
pip install --quiet torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.4 MB/s eta 0:00:00


In [10]:
!python scripts/colab_prepare_congestion_data.py --num-samples 500 --no-eval



=== macro_region ===
downloading...
Downloading...
From: https://drive.google.com/uc?id=14n9khpSK56NUZrGUNPPRYmZkAbCOstKq
To: /content/circuitnet_downloads/macro_region.tar.gz
100% 6.10M/6.10M [00:00<00:00, 107MB/s]
picked 500 sample IDs from macro_region
/content/closed-loop-placement/scripts/colab_prepare_congestion_data.py:173: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, extract_dir)
extracted 500 files from macro_region
deleted /content/circuitnet_downloads/macro_region.tar.gz to free space

=== rudy ===
downloading...
Downloading...
From (original): https://drive.google.com/uc?id=1KUocSofLvyAFKXu8AXt4j3TPJsiaCjS6
From (redirected): https://drive.google.com/uc?id=1KUocSofLvyAFKXu8AXt4j3TPJsiaCjS6&confirm=t&uuid=e19d7ca1-b594-4c94-aabf-a982e6210a45
To: /content/circuitnet_downloads/rudy.tar.gz
100% 2.73G/2.73G [00:30<00:00, 90.4MB/s]

In [11]:
!python scripts/preprocess_circuitnet.py \
    --root data/circuitnet_raw/congestion \
    --out /content/circuitnet_graphs \
    --limit 500


Done. 500 graphs written to /content/circuitnet_graphs, 0 skipped.


In [25]:
import importlib, gnn.model
importlib.reload(gnn.model)
from gnn.model import CongestionGNN


ckpt = torch.load("checkpoints/pretrained.pt", map_location="cpu")
model = CongestionGNN(in_channels=5)
model.load_state_dict(ckpt["model"] if "model" in ckpt else ckpt)
model.eval()

g = torch.load(glob.glob("/content/circuitnet_graphs/*.pt")[0], weights_only=False)
with torch.no_grad():
    pred = model(g.x, g.edge_index)
print("pred std:", pred.std().item(), "label std:", g.y.std().item())
print("pred mean:", pred.mean().item(), "label mean:", g.y.mean().item())


pred std: 0.01132422499358654 label std: 0.018049227073788643
pred mean: 0.1289944052696228 label mean: 0.14110025763511658


In [24]:
!python scripts/train_pretrain.py \
    --data /content/circuitnet_graphs \
    --epochs 50 \
    --out checkpoints/pretrained.pt \
    --lr 3e-4 \
    --batch-size 2

Loaded 500 graphs
in_channels=5, device=cuda
epoch 001  train_loss=0.0059  val_loss=0.0041
  -> saved best checkpoint to checkpoints/pretrained.pt
epoch 002  train_loss=0.0042  val_loss=0.0042
epoch 003  train_loss=0.0040  val_loss=0.0036
  -> saved best checkpoint to checkpoints/pretrained.pt
epoch 004  train_loss=0.0039  val_loss=0.0037
epoch 005  train_loss=0.0037  val_loss=0.0043
epoch 006  train_loss=0.0037  val_loss=0.0038
epoch 007  train_loss=0.0037  val_loss=0.0040
epoch 008  train_loss=0.0036  val_loss=0.0036
  -> saved best checkpoint to checkpoints/pretrained.pt
epoch 009  train_loss=0.0035  val_loss=0.0039
epoch 010  train_loss=0.0035  val_loss=0.0037
epoch 011  train_loss=0.0035  val_loss=0.0035
  -> saved best checkpoint to checkpoints/pretrained.pt
epoch 012  train_loss=0.0034  val_loss=0.0035
  -> saved best checkpoint to checkpoints/pretrained.pt
epoch 013  train_loss=0.0035  val_loss=0.0036
epoch 014  train_loss=0.0034  val_loss=0.0035
epoch 015  train_loss=0.0034  v

In [ ]:
!git add checkpoints/pretrained.pt
!git commit -m "continue pretraining"
!git push